# RiftLab — CC-701 no Colab

Planejamento do 3º ao 5º item; foco principal no 4º. Compara popularidade,
baseline condicionado à build, CatBoost contextual e ablação sem composição.
Seleção só na validação; uma tentativa de avaliação do teste. Teste já usado em
experimentos anteriores, sem alegar confirmação independente. Nenhuma promoção automática.

1. No Google Drive, crie **riftlab-cc701** e coloque **riftlab-cc701-colab.zip** nela.
2. No Colab: **Ambiente de execução → Alterar tipo de ambiente → GPU T4** (gratuita quando disponível).
3. Execute as células em ordem. Não é necessário deixar o WSL aberto.
4. Devolva **riftlab-cc701-resultados.zip**, gerado na última célula, mesmo se houver falha.

O pacote não contém raw, credenciais nem IDs originais de partidas/jogadores.
O treino é remoto; a matriz tem preflight de memória e armazenamento float32.
Se a sessão cair, os checkpoints ficam no Drive. Rode novamente: checkpoints
concluídos são conferidos e reaproveitados. Se o teste tiver começado e não
terminado, o programa recusa repeti-lo; gere o ZIP parcial e devolva para análise.


In [ ]:
# 1. Drive, dispositivo e pacote verificado
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, pathlib, subprocess, sys, zipfile, datetime
BASE = pathlib.Path('/content/drive/MyDrive/riftlab-cc701')
ZIP = BASE / 'riftlab-cc701-colab.zip'
DEVICE = 'GPU'  # Se não houver GPU gratuita: opção explícita 'CPU', ainda no Colab.
assert ZIP.exists(), f'Coloque o ZIP em {ZIP}'
EXPECTED = 'f9e473bd1c99b4ad98dae60f5b35ea75ac44ec2b7bd344ed2629b4bd4da646ca'
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(ZIP) == EXPECTED, 'ZIP divergente: não executar'
BUNDLE = pathlib.Path('/content/riftlab-cc701-bundle')
BUNDLE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None, 'CRC inválido'
    for name in archive.namelist():
        path = pathlib.PurePosixPath(name)
        assert not path.is_absolute() and '..' not in path.parts
    archive.extractall(BUNDLE)
manifest = json.loads((BUNDLE / 'MANIFEST.json').read_text())
assert all(sha(BUNDLE / name) == digest for name, digest in manifest['files'].items())
OUT = BASE / f'out-v1-{DEVICE.lower()}'
OUT.mkdir(parents=True, exist_ok=True)
print('Pacote íntegro:', len(manifest['files']), 'arquivos. Dispositivo:', DEVICE)
if DEVICE == 'GPU':
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)


In [ ]:
# 2. Dependências e preflight — processo novo evita misturar NumPy do kernel
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BUNDLE / 'requirements-colab.txt')], check=True)
env = dict(os.environ, PYTHONPATH=str(BUNDLE / 'src'))
subprocess.run([sys.executable, '-c', "import catboost,numpy; print('catboost',catboost.__version__,'numpy',numpy.__version__)"], check=True)
COMMAND = [sys.executable, '-u', str(BUNDLE / 'train_colab.py'), str(BUNDLE), str(OUT), '--device', DEVICE]
subprocess.run(COMMAND + ['--check-only'], env=env, check=True)


In [ ]:
# 3. Treino remoto e teste único. Log e checkpoints persistidos no Drive.
with (OUT / 'execution.log').open('a') as log:
    stamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
    log.write('\nLANÇAMENTO ' + stamp + '\n'); log.flush()
    proc = subprocess.Popen(COMMAND, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=''); log.write(line); log.flush()
    status = proc.wait()
(OUT / 'last-launch-status.json').write_text(json.dumps({'exitCode':status,'utc':stamp}) + '\n')
print('Execução concluída.' if status == 0 else 'Execução falhou: gere o ZIP parcial na próxima célula e devolva; não altere a grade.')


In [ ]:
# 4. ZIP de retorno — também funciona com execução parcial/interrompida
files = sorted(p for p in OUT.rglob('*') if p.is_file() and p.name != 'RETURN-MANIFEST.json')
return_manifest = {'card':'CC-701','inputZipSha256':sha(ZIP),
    'inputManifestSha256':sha(BUNDLE / 'MANIFEST.json'),
    'complete':(OUT / 'result.json').exists(),
    'files':{str(p.relative_to(OUT)):sha(p) for p in files}}
(OUT / 'RETURN-MANIFEST.json').write_text(json.dumps(return_manifest, indent=1, sort_keys=True) + '\n')
ARCHIVE = BASE / 'riftlab-cc701-resultados.zip'
if ARCHIVE.exists():
    ARCHIVE = BASE / ('riftlab-cc701-resultados-' + datetime.datetime.now().strftime('%Y%m%dT%H%M%S') + '.zip')
with zipfile.ZipFile(ARCHIVE, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file(): archive.write(path, str(path.relative_to(OUT)))
print('Gerado:', ARCHIVE)
print('Resultado completo:', return_manifest['complete'])
from google.colab import files as colab_files
colab_files.download(str(ARCHIVE))
